[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syaikhipin/kdd26-memdiag/blob/main/notebooks/5_full_real_benchmark.ipynb)

Open this notebook in Google Colab for a rendered, runnable tutorial view: https://colab.research.google.com/github/syaikhipin/kdd26-memdiag/blob/main/notebooks/5_full_real_benchmark.ipynb


# 5. Full real-dataset benchmark

This notebook runs the benchmark command used for the KDD tutorial demo.

For a full local run, use all datasets below. For Colab, consider `--max-items 100` or only `longmemeval_oracle.json`.

## Proposal benchmarking design

This notebook implements the standardized benchmarking pipeline:

1. **INGEST:** load experimental data into memory records.
2. **INDEX:** build lexical/offline retrieval indices through `MemoryStore`.
3. **SEARCH:** execute queries and retrieve relevant context.
4. **ANSWER:** produce evidence-available or insufficient-evidence decisions.
5. **EVALUATE:** compare retrieved IDs against ground-truth evidence/context IDs and optionally run semantic evaluators.
6. **REPORT:** aggregate precision, recall, hit rate, utilization, latency, cost, failure modes, and semantic scores.

Provider mapping in this implementation:
- Memory providers: verbatim, extracted facts, episodic, hybrid.
- Evaluators: offline evidence-ID evaluator plus optional offline semantic, Rhesis, and Semantica backends.
- Benchmarks: LoCoMo, LongMemEval, MemoryArena. LCBench/HPOBench are optional extensions.

In [ ]:
import sys
from pathlib import Path

def _find_pkg_dir(start):
 cur = Path(start).resolve()
 for cand in [cur, *cur.parents]:
 for sub in ("source", "experiment"):
 if (cand / sub / "run.py").exists():
 return cand / sub
 return None

PROJECT_ROOT = Path.cwd()
EXPERIMENT_DIR = _find_pkg_dir(PROJECT_ROOT)
if EXPERIMENT_DIR is None:
 raise FileNotFoundError("Could not locate source/run.py (or source/run.py). Run this notebook from the repository root, or `pip install -e source` first.")
PROJECT_ROOT = EXPERIMENT_DIR.parent
RESULTS_DIR = PROJECT_ROOT / "results"
sys.path.insert(0, str(EXPERIMENT_DIR))
print("EXPERIMENT_DIR =", EXPERIMENT_DIR)
print("Experiment code exists:", (EXPERIMENT_DIR / "run.py").exists())


In [ ]:
import subprocess
cmd = [
 sys.executable, str(EXPERIMENT_DIR / "run.py"),
 "--mode", "real",
 "--backend", "offline",
 "--datasets", "locomo", "longmemeval", "memoryarena",
 "--max-conversations", "999",
 "--max-questions", "999",
 "--top-k", "5",
 "--eval-backend", "offline",
 "--eval-limit", "50",
 "--visualize",
]
print(" ".join(map(str, cmd)))
# To keep notebook reruns fast, only execute if RUN_FULL_BENCHMARK=1.
if os.environ.get("RUN_FULL_BENCHMARK") == "1":
 result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True, timeout=600)
 print(result.stdout)
 print(result.stderr)
 assert result.returncode == 0
else:
 print("Skipped full benchmark. Set RUN_FULL_BENCHMARK=1 to execute.")

In [ ]:
modal_cmd = [
 sys.executable, str(EXPERIMENT_DIR / "run.py"),
 "--runner", "modal",
 "--modal-gpu", os.environ.get("MODAL_GPU", "T4"),
 "--modal-detach",
 "--mode", "real",
 "--backend", "offline",
 "--datasets", "locomo", "longmemeval", "memoryarena",
 "--max-conversations", "999",
 "--max-questions", "999",
 "--top-k", "5",
 "--eval-backend", "offline",
 "--visualize",
]
print("Paper-quality Modal GPU command:")
print(" ".join(map(str, modal_cmd)))
if os.environ.get("RUN_MODAL_FULL_BENCHMARK") == "1":
 result = subprocess.run(modal_cmd, cwd=PROJECT_ROOT, text=True, capture_output=True, timeout=3600)
 print(result.stdout)
 print(result.stderr)
 assert result.returncode == 0
else:
 print("Skipped Modal benchmark. Set RUN_MODAL_FULL_BENCHMARK=1 after Modal setup to execute.")
 print("Detached Modal runs print a call id; fetch with --modal-call-id <call-id>.")

In [ ]:
print("Optional external evaluator smoke commands:")
print("""
# Rhesis: set RHESIS_API_KEY outside the notebook first.
python source/run.py \\
 --mode real \\
 --eval-backend rhesis \\
 --eval-smoke-test \\
 --eval-limit 1

# Semantica: requires the semantica package to be installed.
python source/run.py \\
 --mode real \\
 --eval-backend semantica \\
 --eval-smoke-test \\
 --eval-limit 1
""")

In [ ]:
latest = sorted(RESULTS_DIR.glob("run_*_real_metrics.json"))[-1]
print("Reading", latest)
metrics = json.loads(latest.read_text())["real"]
for dataset, summary in metrics["datasets"].items():
 for strategy, row in summary["by_strategy"].items():
 print(dataset, strategy, row["questions"], row["retrieval_precision"], row["retrieval_recall"], row["evidence_hit_rate"])

Guidance: use this for Exercise 2. Participants compare strategies across datasets and explain why no single architecture dominates.